# Sensibilidad química de theta_nv2 y sus efectos en CO₂ y azúcares
Sólo theta_nv2, con un parámetro cambiado por escenario. No hay optimización ni se sustituyen parámetros guardados. Los valores de KIE (122.206 y 24.895 g/L) y KIG (17.845 g/L) proceden de las comparaciones previas. Las tasas máximas se ensayan ±20% para explorar ambos sentidos sin trasladar de golpe la betaF0 de theta_full, que es unas 7.7 veces mayor.

Todos los escenarios conservan los rendimientos originales de theta_nv2, incluidos Yeg y Yef ≤0.5114. Misma temperatura, iniciales y nutriciones por LAB. LAB019–021 conserva la corrección de concentración de dosis por volumen remanente usada en el diagnóstico anterior.

CO₂: se conserva la formulación histórica, con sus modulaciones y depósito, usando los mismos coeficientes antiguos para todas las curvas. Cd inicial cero, sin filtrado de la predicción. No se aplica onset manual. La ganancia no se reajusta para ocultar cambios de amplitud. Las referencias experimentales de CO₂ son medianas horarias de RAW convertida por volumen remanente; G+F usa su ensayo conjunto y YAN NUT no se excluye de los datos disponibles. LAB016–018 sólo muestra química cuando hay mediciones disponibles en la fuente; no se inventan observaciones.

Este ensayo es una sensibilidad, no una nueva calibración ni selección de parámetros. Se informan los fallos de integración si un escenario sale del dominio del modelo original.

KIE es la constante de inhibición por etanol: al reducirla aumenta esa inhibición. KIG controla la inhibición que la glucosa ejerce sobre el consumo de fructosa: al reducirla aumenta esa inhibición. betaG0 y betaF0 son las tasas fermentativas máximas de glucosa y fructosa a la temperatura de referencia.

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys,json,hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/sources.csv').exists())
sys.dont_write_bytecode=True
sys.path.insert(0,str(ROOT/'src'))
assert Path(sys.prefix).name.lower()=='fermentation'
from wine_fermentation.models.historical.calibration import load,release_forward,GAS_NAMES
from wine_fermentation.comparison import observations,chemical_observations,LABS

def hashes():
    return {str(p):hashlib.sha256(p.read_bytes()).hexdigest() for folder in ('src','configs','results','data/compatibility')
            for p in (ROOT/folder).rglob('*') if p.is_file() and '__pycache__' not in p.parts}
protected=hashes()
V=load(ROOT)
base=V.thetas['theta_nv2'].copy()
data=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',usecols=['lab','time_h','record_kind','remaining_volume_L','co2_raw_g_l_h'])
obs_co2=observations(data)
for lab in LABS[3:]:
    events=data[(data.lab==lab)&(data.record_kind=='withdrawal_after')].sort_values('time_h')
    et=events.time_h.to_numpy();ev=events.remaining_volume_L.to_numpy()
    def volume(t):return np.r_[2.,ev][np.searchsorted(et,t,side='right')]
    batch=V.batches[lab]
    V.batches[lab]=replace(batch,pulses={**batch.pulses,'N':tuple((t,d*2/float(volume(t))) for t,d in batch.pulses['N'])})
reference=pd.read_csv(ROOT/'configs/co2_reference_parameters.csv',float_precision='round_trip')
row=reference[(reference.theta=='theta_full')&(reference.training=='train_019_021')&(reference.stage=='Z2')&(reference['mode']=='v2')&(reference.filtering=='offline_30min')]
assert len(row)==1
row=row.iloc[0]
gas={**V.old,**{k:float(row[k]) for k in GAS_NAMES}}
display(pd.Series({k:gas[k] for k in GAS_NAMES},name='Coeficientes CO₂ fijos'))
changes={'theta_nv2':{},'KIE = 122.206':{'Kie0':122.20601783644179},'KIE = 24.895':{'Kie0':24.895475735199245},
         'KIG = 17.845':{'Kig0':17.84521292773551},'betaG −20%':{'betaG0':.8*base['betaG0']},
         'betaG +20%':{'betaG0':1.2*base['betaG0']},'betaF −20%':{'betaF0':.8*base['betaF0']},
         'betaF +20%':{'betaF0':1.2*base['betaF0']}}
parameters={name:{**base,**change} for name,change in changes.items()}
for name,p in parameters.items():
    assert 0<p['Yeg']<=.5114 and 0<p['Yef']<=.5114
    assert all(p[k]==base[k] for k in ('Yeg','Yef','Yxn','Yxg','Yxf'))
    assert sum(p[k]!=base[k] for k in base)<=1
display(pd.DataFrame([dict(scenario=name,**{k:p[k] for k in ('Kie0','Kig0','betaG0','betaF0','Yeg','Yef')}) for name,p in parameters.items()]))


In [ ]:
curves={};cores={};failures=[];metrics=[];early=[]
for name,p in parameters.items():
    for lab in LABS:
        try:
            V.prepare(name,p,(lab,))
            core=V.cores[name,lab];cache=V.cache[name,lab]
            prediction,cd,qgas=release_forward(cache,gas)
            curve=pd.DataFrame(dict(qpred=prediction,Cd=cd,qgas=qgas,qprod=cache['source']),index=cache['time'])
            assert np.isfinite(prediction).all() and cd.min()>=0
            curves[name,lab]=curve;cores[name,lab]=core
        except (RuntimeError,ValueError,FloatingPointError) as error:
            failures.append(dict(scenario=name,lab=lab,error=str(error)));continue
        cut=V.horizons[lab]
        for channel in ('CO2','E','G','F','GF'):
            if channel=='CO2':
                o=obs_co2[lab].rename(columns={'time_h':'t_h','co2_raw_g_l_h':'value'});f=curve.qpred
            else:
                o=chemical_observations(V.chem,lab,channel,after_initial=True)
                col={'E':'E_g_l','G':'G_g_l','F':'F_g_l','GF':'GF_g_l'}[channel]
                o=o.rename(columns={col:'value'});f=core.G+core.F if channel=='GF' else core[channel]
            o=o[o.t_h.between(0,min(cut,float(f.index.max())))]
            if len(o):
                predicted=np.interp(o.t_h,f.index,f)
                e=predicted-o.value.to_numpy()
                lod=np.where(o.value.to_numpy()>=.05,e,np.maximum(predicted-.05,0.)) if channel=='CO2' else e
                metrics.append(dict(scenario=name,lab=lab,variable=channel,n=len(o),RMSE=float(np.sqrt(np.mean(e**2))),bias=float(e.mean()),objective_MSE=float(np.mean(lod**2))))
        pulse=V.batches[lab].pulses['N'][0][0]
        if pulse>55:
            a,b=np.interp([35.,55.],curve.index,curve.qprod)
            early.append(dict(scenario=name,lab=lab,source35=a,source55=b,decline_percent=100*(1-b/a)))
    print('Simulado:',name)
metrics=pd.DataFrame(metrics);failures=pd.DataFrame(failures)
display(Markdown('**Fallos de integración:** se conservan explícitamente; no se recortan estados ni se cambian ecuaciones para resolverlos.'))
display(failures if len(failures) else pd.DataFrame({'estado':['Ningún fallo de integración']}))
display(Markdown('**Error por LAB y variable:** cada escenario se compara con las mismas observaciones disponibles. CO₂ usa pérdida con umbral 0.05; el RMSE ordinario se muestra por separado.'))
with pd.option_context('display.max_rows',200):
    display(metrics)
display(Markdown('**Caída de producción antes de nutrición entre 35 y 55 h:** sólo LAB cuyo pulso ocurre después de 55 h.'))
display(pd.DataFrame(early).pivot(index='scenario',columns='lab',values='decline_percent'))


## Predicciones y mediciones
Cada figura tiene una variable y seis LAB. Todas las líneas de escenarios provienen de theta_nv2. La referencia negra usa el ensayo conjunto para G+F y mediciones separadas cuando existen para G/F. El CO₂ medido se dibuja como línea de medianas; las mediciones químicas discretas se muestran como muestras. Las líneas verticales marcan las nutriciones sin moverlas. Las curvas se limitan al horizonte original de evaluación de cada LAB; su elección es común a todos los escenarios.

In [ ]:
palette=['#0072B2','#D55E00','#56B4E9','#CC79A7','#E69F00','#999933','#009E73','#882255']
colors=dict(zip(parameters,palette))
for channel,title in [('CO2','CO₂ emitido'),('E','Etanol'),('G','Glucosa'),('F','Fructosa'),('GF','Glucosa + fructosa')]:
    fig,axes=plt.subplots(2,3,figsize=(17,10))
    fig.subplots_adjust(top=.72,bottom=.08,hspace=.40,wspace=.26)
    fig.suptitle(title+' · sensibilidad de theta_nv2\nUn parámetro por escenario · rendimientos y capa CO₂ fijos',y=.99,fontsize=14)
    for ax,lab in zip(axes.flat,LABS[3:]+LABS[:3]):
        cut=V.horizons[lab]
        if channel=='CO2':
            o=obs_co2[lab];o=o[o.time_h.between(0,cut)]
            ax.plot(o.time_h,o.co2_raw_g_l_h,color='black',lw=1.3,label='Mediciones',zorder=6)
        else:
            o=chemical_observations(V.chem,lab,channel)
            col={'E':'E_g_l','G':'G_g_l','F':'F_g_l','GF':'GF_g_l'}[channel]
            ax.scatter(o.t_h,o[col],color='black',s=15,label='Mediciones',zorder=6)
        for name in parameters:
            if (name,lab) not in cores:continue
            core=cores[name,lab];f=curves[name,lab].qpred if channel=='CO2' else core.G+core.F if channel=='GF' else core[channel]
            keep=f.index<=cut
            ax.plot(f.index[keep],f[keep],color=colors[name],lw=2.1 if name=='theta_nv2' else 1.3,
                    ls='-' if name=='theta_nv2' else '--',label=name)
        for event,dose in V.batches[lab].pulses['N']:ax.axvline(event,color='gray',ls=':',lw=.8)
        ax.set_title(lab);ax.set_xlim(0,cut);ax.set_ylim(bottom=0);ax.set_xlabel('Tiempo desde inoculación [h]')
        ax.set_ylabel('g CO₂/(L·h)' if channel=='CO2' else 'g/L');ax.grid(alpha=.25)
    h,l=axes[0,0].get_legend_handles_labels()
    fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,.90),ncol=3,title='Escenarios químicos · línea vertical: nutrición',fontsize=10)
    plt.show()


## Comparación con la referencia y verificaciones
Un RMSE menor no implica mejora general: comprobar conjuntamente las cinco variables y ambos grupos de laboratorios. Esta sensibilidad utiliza datos ya explorados y no se presenta como una validación nueva. No se elige ni guarda un nuevo theta.

In [ ]:
reference_metrics=metrics[metrics.scenario=='theta_nv2'][['lab','variable','RMSE']].rename(columns={'RMSE':'baseline_RMSE'})
comparison=metrics.merge(reference_metrics,on=['lab','variable'],validate='many_to_one')
comparison['delta_RMSE']=comparison.RMSE-comparison.baseline_RMSE
comparison['grupo']=np.where(comparison.lab.isin(LABS[3:]),'LAB019–021','LAB016–018')
display(comparison.groupby(['scenario','grupo','variable']).delta_RMSE.mean().unstack('variable'))
endpoints=[]
for (name,lab),core in cores.items():
    t=min(V.horizons[lab],core.index.max())
    endpoints.append(dict(scenario=name,lab=lab,time_h=t,E=float(np.interp(t,core.index,core.E)),G=float(np.interp(t,core.index,core.G)),F=float(np.interp(t,core.index,core.F))))
display(pd.DataFrame(endpoints))
assert protected==hashes(),'Se modificó un archivo protegido'
assert len(curves)+len(failures)==len(parameters)*len(LABS)
print('Modelos, parámetros guardados, results y datos de compatibilidad intactos. Escenarios sólo en memoria; rendimientos conservados.')

counts={lab:len(chemical_observations(V.chem,lab,"GF")) for lab in LABS[3:]}
assert list(counts.values())==[12,12,10],counts
display(pd.Series(counts,name="Muestras G+F disponibles, incluida inicial"))
display(Markdown("**Lectura del ensayo:** reducir KIE acentúa la caída de producción en LAB016–018, pero también frena la fermentación y deja más azúcares al horizonte común. El cambio de KIG puede reducir la amplitud del CO₂ sin recuperar esa caída: son efectos distintos. Los escenarios no representan nuevos ajustes aceptados. LAB016–018 no tiene observaciones de E/G/F/G+F en la tabla química cargada; allí las figuras comparan predicciones y no permiten evaluar error químico experimental."))
